# Raw Dataset Processor

Pipeline monitoring untuk mengubah raw dataset `datasets/url_discovery/*.json` menjadi kandidat dataset dengan schema seperti `datasets/v1_shs_datasets.csv`. Notebook ini tidak menyimpan output secara otomatis.

## 1. Setup

In [113]:
from pathlib import Path
import sys

import polars as pl
from IPython.display import display


def find_project_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in (current, *current.parents):
        if (candidate / "config.py").exists() and (candidate / "services").is_dir():
            return candidate
    raise FileNotFoundError("Root proyek tidak ditemukan")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import config
from services.dataset_service import DatasetService

pl.Config.set_tbl_hide_column_data_types(True)
pl.Config.set_tbl_cell_alignment("LEFT")
pl.Config.set_fmt_str_lengths(250)
pl.Config.set_tbl_width_chars(180)
pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_rows(20)

dataset_service = DatasetService()
RAW_FOLDER = config.DATASETS / "url_discovery"
RESEARCH_CONFIG_PATH = PROJECT_ROOT / "research_config.json"
REFERENCE_DATASET_PATH = config.DATASETS / "v1_shs_datasets.csv"

print(f"Raw folder: {RAW_FOLDER}")
print(f"Research config: {RESEARCH_CONFIG_PATH}")
print(f"Reference schema: {REFERENCE_DATASET_PATH}")

Raw folder: E:\School\tugas-akhir\project\datasets\url_discovery
Research config: E:\School\tugas-akhir\project\research_config.json
Reference schema: E:\School\tugas-akhir\project\datasets\v1_shs_datasets.csv


## 2. Load Config dan Raw Discovery

In [114]:
research_config = dataset_service.load_research_config(RESEARCH_CONFIG_PATH)
meta_df = dataset_service.load_url_discovery_meta(RAW_FOLDER)
queries_df = dataset_service.load_url_discovery_queries(RAW_FOLDER)
raw_records_df = dataset_service.load_url_discovery_records(RAW_FOLDER)

RECORDS_FILTER_CONDITIONS = [
    # pl.col("content_status") == "success",
    # pl.col("content_text").fill_null("").str.strip_chars().str.len_chars() > 0,
    # ~pl.col("domain").str.contains("instagram.com|tiktok.com|facebook.com"),
]


def apply_filter_conditions(df: pl.DataFrame, conditions: list) -> pl.DataFrame:
    if not conditions:
        return df
    expression = conditions[0]
    for condition in conditions[1:]:
        expression = expression & condition
    return df.filter(expression)


records_df = apply_filter_conditions(raw_records_df, RECORDS_FILTER_CONDITIONS)

print("Search label:", research_config.get("search_label"))
print("Primary location:", research_config.get("primary_location"))
print(f"File metadata: {meta_df.height:,}")
print(f"Query rows: {queries_df.height:,}")
print(f"Raw records unik: {raw_records_df.height:,}")
print(f"Records setelah filter manual: {records_df.height:,}")

display(meta_df)

Search label: SHS/PLTS Kalimantan Barat
Primary location: Kalimantan Barat
File metadata: 7
Query rows: 1,577
Raw records unik: 1,105
Records setelah filter manual: 1,105


source_file,built_at,search_label,source_files,record_count,content_count,query_count
"""dataset_20260627T094855Z.json""","""2026-06-27T09:48:55.240893+00:00""","""SHS/PLTS Kalimantan Barat""","{""E:\Work\personal\url-discovery\data\urls.json"",""E:\Work\personal\url-discovery\data\contents.json"",""E:\Work\personal\url-discovery\data\search_state.json""}",214,214,335
"""dataset_20260627T101253Z.json""","""2026-06-27T10:12:53.267116+00:00""","""SHS/PLTS Kalimantan Barat""","{""E:\Work\personal\url-discovery\data\urls.json"",""E:\Work\personal\url-discovery\data\contents.json"",""E:\Work\personal\url-discovery\data\search_state.json""}",203,203,268
"""dataset_20260627T135259Z.json""","""2026-06-27T13:52:59.725144+00:00""","""SHS/PLTS Kalimantan Barat""","{""E:\Work\personal\url-discovery\data\urls.json"",""E:\Work\personal\url-discovery\data\contents.json"",""E:\Work\personal\url-discovery\data\search_state.json""}",88,88,197
"""dataset_20260628T033631Z.json""","""2026-06-28T03:36:31.219965+00:00""","""SHS/PLTS Kalimantan Barat""","{""E:\Work\personal\url-discovery\data\urls.json"",""E:\Work\personal\url-discovery\data\contents.json"",""E:\Work\personal\url-discovery\data\search_state.json""}",202,202,381
"""dataset_20260628T104512Z.json""","""2026-06-28T10:45:12.810124+00:00""","""SHS/PLTS Kalimantan Barat""","{""E:\Work\personal\url-discovery\data\urls.json"",""E:\Work\personal\url-discovery\data\contents.json"",""E:\Work\personal\url-discovery\data\search_state.json""}",136,136,369
"""dataset_20260628T113525Z.json""","""2026-06-28T11:35:25.172805+00:00""","""SHS/PLTS Kalimantan Barat""","{""E:\Work\personal\url-discovery\data\urls.json"",""E:\Work\personal\url-discovery\data\contents.json"",""E:\Work\personal\url-discovery\data\search_state.json""}",12,12,28
"""dataset_20260628T173013Z.json""","""2026-06-28T17:30:13.810625+00:00""","""SHS/PLTS Kalimantan Barat""","{""E:\Work\personal\url-discovery\data\urls.json"",""E:\Work\personal\url-discovery\data\contents.json"",""E:\Work\personal\url-discovery\data\search_state.json""}",250,250,267


## 3. Monitoring Kualitas Raw Dataset

In [115]:
status_distribution = (
    records_df.with_columns(pl.col("content_status").fill_null("missing"))
    .group_by("content_status")
    .agg(pl.len().alias("jumlah"))
    .sort("jumlah", descending=True)
)

domain_distribution = (
    records_df.with_columns(pl.col("domain").fill_null("unknown"))
    .group_by("domain")
    .agg(pl.len().alias("jumlah"))
    .sort("jumlah", descending=True)
    .head(20)
)

query_group_distribution = (
    records_df.with_columns(pl.col("query_group").fill_null("unknown"))
    .group_by("query_group")
    .agg(pl.len().alias("jumlah"))
    .sort("jumlah", descending=True)
    .head(20)
)

text_length_summary = records_df.select(
    pl.len().alias("total_records"),
    (
        pl.col("content_text")
        .fill_null("")
        .str.strip_chars()
        .str.len_chars()
        > 0
    ).sum().alias("records_with_text"),
    pl.col("content_text_length").min().alias("min_text_length"),
    pl.col("content_text_length").mean().round(2).alias("avg_text_length"),
    pl.col("content_text_length").max().alias("max_text_length"),
)

display(status_distribution)
display(domain_distribution)
display(query_group_distribution)
display(text_length_summary)

content_status,jumlah
"""success""",872
"""too_short""",147
"""pdf_skipped""",45
"""failed""",41


domain,jumlah
"""www.instagram.com""",428
"""www.tiktok.com""",137
"""www.facebook.com""",50
"""pontianakpost.jawapos.com""",30
"""rri.co.id""",25
"""kalbar.antaranews.com""",25
"""www.researchgate.net""",9
"""id.scribd.com""",8
"""hijau.bisnis.com""",8
"""gatrik.esdm.go.id""",8


query_group,jumlah
"""core:electricity_access""",232
"""issue:benefit""",93
"""issue:problem""",84
"""social""",75
"""issue:access""",62
"""issue:environment""",55
"""issue:experience""",50
"""entity-opinion""",48
"""issue:socioeconomic""",47
"""issue:funding""",46


total_records,records_with_text,min_text_length,avg_text_length,max_text_length
1105,897,0,3252.8,33760


## 4. Bangun Kandidat Schema v1

In [116]:
candidate_df = dataset_service.build_v1_candidate_rows(
    records_df=records_df,
    research_config=research_config,
)

CANDIDATE_FILTER_CONDITIONS = [
    pl.col("location") == "",
    # pl.col("source_url").str.to_lowercase().str.contains("facebook.com"),
    # ~pl.col("text").str.to_lowercase().str.contains("penayangan"),
    # pl.col("aspect").is_in(["experience", "benefit", "access"]),
]

filtered_candidate_df = apply_filter_conditions(candidate_df, CANDIDATE_FILTER_CONDITIONS)

reference_df = dataset_service.load(REFERENCE_DATASET_PATH)
expected_columns = list(dataset_service.v1_dataset_columns())
missing_candidate_columns = [column for column in expected_columns if column not in filtered_candidate_df.columns]
extra_candidate_columns = [column for column in filtered_candidate_df.columns if column not in expected_columns]

print(f"Candidate rows sebelum filter kandidat: {candidate_df.height:,}")
print(f"Candidate rows setelah filter kandidat: {filtered_candidate_df.height:,}")
print("Kolom wajib hilang:", missing_candidate_columns)
print("Kolom monitoring tambahan:", extra_candidate_columns)
print("Urutan kolom utama sama:", filtered_candidate_df.columns[: len(expected_columns)] == expected_columns)

display(filtered_candidate_df.select(expected_columns).head(20))

Candidate rows sebelum filter kandidat: 1,105
Candidate rows setelah filter kandidat: 143
Kolom wajib hilang: []
Kolom monitoring tambahan: ['raw_source_file', 'raw_domain', 'content_status', 'query_group', 'query', 'raw_title', 'raw_text_length']
Urutan kolom utama sama: True


text_id,text,subjectivity_type,speaker_type,public_opinion_scope,corpus_role,aspect,location,sentiment_label,label_status,source_id,source_type,source_url,dataset_tier,inclusion_status,verification_status,evidence_support_score,parent_text_id,decision_note
"""RAW-0012""","""TikTok - Make Your Day Program Listrik Desa Perluas Akses Listrik ke Dusun Terpencil TikTok · Dunia Punya Cerita 92 rb+ penayangan · 4 hari yang lalu @duniapunyacerita 7654119865329978644""","""contextual_source""","""community_representative""","""contextual_reference""","""excluded""","""electricity_access""","""""","""""","""unlabeled""","""RAW-SRC-0012""","""online_news""","""https://www.tiktok.com/@duniapunyacerita_/video/7654119865329978644""","""C_holdout_excluded""","""held_out_not_for_sentiment_core""","""perlu_verifikasi""",0.45,"""""","""content_not_success"""
"""RAW-0021""","""Zona Energi Baru Terbarukan on Instagram: ""Pemerintah Provinsi Kalimantan Utara (Pemprov Kaltara) bersiap membangun Pembangkit Listrik Tenaga Surya (PLTS) komunal di 13 titik pada tahun 2026. Proyek infrastruktur hijau ini difokuskan untuk memperluas…","""public_expectation""","""community_representative""","""public_opinion""","""core_public_opinion""","""experience""","""""","""""","""unlabeled""","""RAW-SRC-0021""","""online_news""","""https://www.instagram.com/p/DZhoPyjD9Sw""","""A_candidate_core""","""candidate_analysis_ready""","""perlu_verifikasi""",0.8,"""""","""candidate_from_raw_discovery"""
"""RAW-0022""","""Pertamina New & Renewable Energy on Instagram: ""KDKMP PULAU SEMBUR BAKAL DITENAGAI PLTS DARI PERTAMINA NRE Kolaborasi antar Kementerian Koperasi RI (@kemenkop) dan @pertamina akan segera terwujud. Melalui subholding Pertamina NRE, saat ini tengah dib…","""public_expectation""","""community_representative""","""public_opinion""","""core_public_opinion""","""environment""","""""","""""","""unlabeled""","""RAW-SRC-0022""","""online_news""","""https://www.instagram.com/reel/DZ7QCeEJir5""","""A_candidate_core""","""candidate_analysis_ready""","""perlu_verifikasi""",0.8,"""""","""candidate_from_raw_discovery"""
"""RAW-0029""","""PLN UID SULSELRABAR on Instagram: ""PLN untuk Rakyat, Nyala Listrik dari Inovasi SuperSUN Hadir di Dua Sekolah Terpencil Seko Electrizen, Haru dan sukacita menyelimuti SDN 084 Amballong dan SDN 080 Turong di Luwu Utara, Sulawesi Selatan! ✨ Perjalanan …","""public_expectation""","""community_representative""","""public_opinion""","""core_public_opinion""","""general_shs""","""""","""""","""unlabeled""","""RAW-SRC-0029""","""online_news""","""https://www.instagram.com/p/DMFYEC7yDIe""","""A_candidate_core""","""candidate_analysis_ready""","""perlu_verifikasi""",0.8,"""""","""candidate_from_raw_discovery"""
"""RAW-0051""","""Bahlil Lahadalia di Instagram: ""Hari ini kami meresmikan tiga proyek strategis Merdeka dari Kegelapan yang diselenggarakan di Kabupaten Minahasa, Sulawesi Utara. Ketiga proyek tersebut meliputi: 1. Program Bantuan Pasang Baru Listrik (BPBL) di Kabupa…","""public_experience""","""community_representative""","""public_opinion""","""core_public_opinion""","""general_shs""","""""","""""","""unlabeled""","""RAW-SRC-0051""","""online_news""","""https://www.instagram.com/p/DQZaPxrgcrY?hl=id""","""A_candidate_core""","""candidate_analysis_ready""","""perlu_verifikasi""",0.8,"""""","""candidate_from_raw_discovery"""
"""RAW-0058""","""Kementerian Energi dan Sumber Daya Mineral on Instagram: ""BERKAH ENERGI TERBARUKAN 🍃 #SobatEnergi, Sebelum PLTMH melistriki, warga Kampung Tangsi Jaya mengandalkan lampu minyak dan obor untuk penerangan sehari-hari. Kini, 95 rumah telah menikmati lis…","""public_expectation""","""community_representative""","""public_opinion""","""core_public_opinion""","""experience""","""""","""""","""unlabeled""","""RAW-SRC-0058""","""online_news""","""https://www.instagram.com/reel/DYFCZ9hvgXq""","""A_candidate_core""","""candidate_analysis_ready""","""perlu_verifikasi""",0.8,"""""","""candidate_from_raw_discov

## 5. Monitoring Kandidat

In [117]:
tier_distribution = (
    filtered_candidate_df.group_by("dataset_tier")
    .agg(pl.len().alias("jumlah"))
    .sort("jumlah", descending=True)
)

inclusion_distribution = (
    filtered_candidate_df.group_by("inclusion_status")
    .agg(pl.len().alias("jumlah"))
    .sort("jumlah", descending=True)
)

source_type_distribution = (
    filtered_candidate_df.group_by("source_type")
    .agg(pl.len().alias("jumlah"))
    .sort("jumlah", descending=True)
)

aspect_distribution = (
    filtered_candidate_df.group_by("aspect")
    .agg(pl.len().alias("jumlah"))
    .sort("jumlah", descending=True)
    .head(20)
)


location_distribution = (
    filtered_candidate_df.group_by("location")
    .agg(pl.len().alias("jumlah"))
    .sort("jumlah", descending=True)
    .head(20)
)

display(tier_distribution)
display(inclusion_distribution)
display(source_type_distribution)
display(aspect_distribution)
display(location_distribution)

dataset_tier,jumlah
"""A_candidate_core""",110
"""C_holdout_excluded""",33


inclusion_status,jumlah
"""candidate_analysis_ready""",110
"""held_out_not_for_sentiment_core""",33


source_type,jumlah
"""online_news""",132
"""academic_repository""",7
"""government_official""",4


aspect,jumlah
"""experience""",62
"""general_shs""",43
"""electricity_access""",12
"""environment""",12
"""benefit""",6
"""grid_hybrid""",5
"""access""",2
"""problem""",1


location,jumlah
"""""",143


## 6. Review Kandidat Prioritas

In [118]:
review_columns = [
    "text_id",
    "dataset_tier",
    "inclusion_status",
    "evidence_support_score",
    "aspect",
    "location",
    "source_type",
    "source_url",
    "text",
    "decision_note",
    "query_group",
    "query",
]

priority_df = (
    filtered_candidate_df
    .filter(pl.col("dataset_tier").is_in(["A_candidate_core", "B_review_queue"]))
    .sort(["dataset_tier", "evidence_support_score"], descending=[False, True])
    .select(review_columns)
)

display(priority_df.head(50))

text_id,dataset_tier,inclusion_status,evidence_support_score,aspect,location,source_type,source_url,text,decision_note,query_group,query
"""RAW-0021""","""A_candidate_core""","""candidate_analysis_ready""",0.8,"""experience""","""""","""online_news""","""https://www.instagram.com/p/DZhoPyjD9Sw""","""Zona Energi Baru Terbarukan on Instagram: ""Pemerintah Provinsi Kalimantan Utara (Pemprov Kaltara) bersiap membangun Pembangkit Listrik Tenaga Surya (PLTS) komunal di 13 titik pada tahun 2026. Proyek infrastruktur hijau ini difokuskan untuk memperluas…","""candidate_from_raw_discovery""","""entity-opinion""","""""PLTS"" ""komentar warga"" ""Dusun Permit"" after:2024-01-01 before:2026-06-27"""
"""RAW-0022""","""A_candidate_core""","""candidate_analysis_ready""",0.8,"""environment""","""""","""online_news""","""https://www.instagram.com/reel/DZ7QCeEJir5""","""Pertamina New & Renewable Energy on Instagram: ""KDKMP PULAU SEMBUR BAKAL DITENAGAI PLTS DARI PERTAMINA NRE Kolaborasi antar Kementerian Koperasi RI (@kemenkop) dan @pertamina akan segera terwujud. Melalui subholding Pertamina NRE, saat ini tengah dib…","""candidate_from_raw_discovery""","""entity-opinion""","""""PLTS"" ""komentar warga"" ""Dusun Permit"" after:2024-01-01 before:2026-06-27"""
"""RAW-0029""","""A_candidate_core""","""candidate_analysis_ready""",0.8,"""general_shs""","""""","""online_news""","""https://www.instagram.com/p/DMFYEC7yDIe""","""PLN UID SULSELRABAR on Instagram: ""PLN untuk Rakyat, Nyala Listrik dari Inovasi SuperSUN Hadir di Dua Sekolah Terpencil Seko Electrizen, Haru dan sukacita menyelimuti SDN 084 Amballong dan SDN 080 Turong di Luwu Utara, Sulawesi Selatan! ✨ Perjalanan …","""candidate_from_raw_discovery""","""core:household_solar""","""""PLTS rumah tangga"" ""Badau"" after:2024-01-01 before:2026-06-27"""
"""RAW-0051""","""A_candidate_core""","""candidate_analysis_ready""",0.8,"""general_shs""","""""","""online_news""","""https://www.instagram.com/p/DQZaPxrgcrY?hl=id""","""Bahlil Lahadalia di Instagram: ""Hari ini kami meresmikan tiga proyek strategis Merdeka dari Kegelapan yang diselenggarakan di Kabupaten Minahasa, Sulawesi Utara. Ketiga proyek tersebut meliputi: 1. Program Bantuan Pasang Baru Listrik (BPBL) di Kabupa…","""candidate_from_raw_discovery""","""issue:access""","""""energi terbarukan"" ""perbatasan tanpa listrik"" ""Paloh"" after:2024-01-01 before:2026-06-27"""
"""RAW-0058""","""A_candidate_core""","""candidate_analysis_ready""",0.8,"""experience""","""""","""online_news""","""https://www.instagram.com/reel/DYFCZ9hvgXq""","""Kementerian Energi dan Sumber Daya Mineral on Instagram: ""BERKAH ENERGI TERBARUKAN 🍃 #SobatEnergi, Sebelum PLTMH melistriki, warga Kampung Tangsi Jaya mengandalkan lampu minyak dan obor untuk penerangan sehari-hari. Kini, 95 rumah telah menikmati lis…","""candidate_from_raw_discovery""","""entity-opinion""","""""tenaga surya"" ""keluhan warga"" ""Dusun Molo"" after:2024-01-01 before:2026-06-27"""
"""RAW-0067""","""A_candidate_core""","""candidate_analysis_ready""",0.8,"""experience""","""""","""online_news""","""https://suarabaru.id/2026/06/22/dari-besek-hingga-masa-depan-anak-listrik-mengubah-kehidupan-warga-desa""","""Dari Besek hingga Masa Depan Anak, Listrik Mengubah Kehidupan Warga Desa - SuaraBaru.id Dari Besek hingga Masa Depan Anak, Listrik Mengubah ... SuaraBaru.id https://suarabaru.id › 2026/06/22 › dari-besek-hingga-... PURWOREJO (SUARABARU.ID) – Bagi Mar…","""candidate_from_raw_discovery""","""entity-opinion""","""""elektrifikasi"" ""respon warga"" ""Desa Jasa"" after:2024-01-01 before:2026-06-27"""
"""RAW-0069""","""A_candidate_core""","""candidate_analysis_ready""",0.8,"""experience""","""""","""online_news""","""https://www.instagram.com/p/DP1lTwJEui2""","""Cabang Dinas ESDM Wilayah VI Tasikmalaya on Instagram: ""Monitoring Pengawasan Pemasangan Instalasi Listrik Desa Program Jabarcaang Kegiatan monitoring dan pengawasan dilakukan untuk memastikan pemasangan instalasi listrik desa melalui program J

## 7. Export Manual Opsional

In [ ]:
EXPORT_CSV = True
EXPORT_PATH = config.OUTPUTS / "datasets" / "un_located_candidate_schema.csv"

if EXPORT_CSV:
    EXPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
    filtered_candidate_df.write_csv(EXPORT_PATH)
    print(f"Candidate dataset terfilter disimpan ke: {EXPORT_PATH}")
else:
    print("Export dimatikan. Ubah EXPORT_CSV = True jika ingin menyimpan kandidat terfilter secara manual.")

Candidate dataset terfilter disimpan ke: E:\School\tugas-akhir\project\outputs\datasets\sekadau_candidate_schema.csv
